# EVolvAI — Physics-Informed Generative Counterfactual VAE
## Research Training Notebook | IEEE Submission Build

---

### Architecture Summary
- **Encoder**: Causal TCN → Multi-Head Self-Attention → Gaussian Posterior
- **Decoder**: Z ⊕ C → FC → TCN decoder
- **Physics Engine**: Differentiable LinDistFlow (V² form) on IEEE 33-Bus radial feeder
- **Data**: NYC PlugNYC EV Sessions + NYC ATVC Traffic + Open-Meteo NYC Weather (2021–2026)

### Google Drive Strategy
All data (raw + processed) lives in Drive → survives runtime disconnects.
| Drive Path | Contents |
|---|---|
| `EVolvAI_Research/data/raw/` | Your raw CSVs (you upload these once) |
| `EVolvAI_Research/data/processed/` | Computed outputs — auto-saved after each step |
| `EVolvAI_Research/checkpoints/` | Model checkpoints every 10 epochs |

### Training Strategy (2-Phase)
| Phase | Epochs | KLD Weight | Physics λ | Purpose |
|---|---|---|---|---|
| Warm-Up | 1-50 | 0 → 0.01 | 0 | Reconstruction focus |
| Physics-On | 51-200 | 0.01 → 0.10 | Activated | LinDistFlow constraints |


## §1 · Environment Setup, Drive Mount & Drive Helpers

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 2>/dev/null || \
!pip install -q torch torchvision
!pip install -q pandas numpy matplotlib scikit-learn pyarrow tqdm openmeteo-requests requests-cache retry-requests

import sys, os, shutil

# ════════════════════════════════════════════════════════════════════════════
# ⚙️  CONFIGURE THIS — folder name inside your Google Drive MyDrive
DRIVE_FOLDER_NAME = "EVolvAI_Research"   # change if your folder is named differently
# ════════════════════════════════════════════════════════════════════════════

IS_COLAB   = False
DRIVE_ROOT = None   # e.g.  /content/drive/MyDrive/EVolvAI_Research

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IS_COLAB   = True
    DRIVE_ROOT = f'/content/drive/MyDrive/{DRIVE_FOLDER_NAME}'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"✅  Colab | Drive mounted → {DRIVE_ROOT}")
except ImportError:
    print("✅  Running locally — Drive sync disabled")

# ── Clone / locate repo ───────────────────────────────────────────────────────
REPO_ROOT = '/content/EVolvAI' if IS_COLAB else os.getcwd()
if IS_COLAB and not os.path.exists(REPO_ROOT):
    !git clone https://github.com/seeramsujay/EVolvAI.git $REPO_ROOT
if IS_COLAB:
    %cd $REPO_ROOT
sys.path.insert(0, REPO_ROOT)
print(f"   CWD        = {os.getcwd()}")
print(f"   DRIVE_ROOT = {DRIVE_ROOT}")

# ── Drive helper functions ────────────────────────────────────────────────────
def drive_path(rel_path):
    """Absolute Drive path for a relative project path. Returns None if not on Colab."""
    return os.path.join(DRIVE_ROOT, rel_path) if DRIVE_ROOT else None

def load_from_drive(local_path, rel_path):
    """
    Copy file from Drive → local if it exists in Drive.
    Returns True if the file was found in Drive (skip recomputing).
    """
    dp = drive_path(rel_path)
    if dp and os.path.exists(dp):
        os.makedirs(os.path.dirname(os.path.abspath(local_path)), exist_ok=True)
        shutil.copy2(dp, local_path)
        size_mb = os.path.getsize(local_path) / 1e6
        print(f"  📂  Loaded from Drive: {rel_path}  ({size_mb:.1f} MB)")
        return True
    return False

def save_to_drive(local_path, rel_path):
    """Copy local file → Drive for persistence across runtimes."""
    dp = drive_path(rel_path)
    if dp and os.path.exists(local_path):
        os.makedirs(os.path.dirname(dp), exist_ok=True)
        shutil.copy2(local_path, dp)
        size_mb = os.path.getsize(dp) / 1e6
        print(f"  💾  Saved to Drive:   {rel_path}  ({size_mb:.1f} MB)")

print("\n✅  Drive helpers ready")


## §2 · Raw Data — Load from Drive
Upload your raw CSVs to Google Drive **once** at:
```
MyDrive/EVolvAI_Research/data/raw/nyc_charging.csv
MyDrive/EVolvAI_Research/data/raw/Automated_Traffic_Volume_Counts_20260415.csv
```
This cell copies them to the Colab local filesystem. If not found in Drive, it falls back to downloading from NYC Open Data.

| Dataset | Schema | Purpose |
|---|---|---|
| `nyc_charging.csv` | `Date, Connected Time, Disconnected Time, Charge Duration (min), Energy Provided (kWh)` | Primary EV sessions 2021–2026 |
| `Automated_Traffic_Volume_Counts_20260415.csv` | `Yr, M, D, HH, MM, Vol` | NYC hourly traffic volume 2021–2026 |


In [ ]:
import os, subprocess, shutil

def ensure_raw_file(local_path, drive_rel, fallback_url, description):
    """Priority: Drive cache → fallback URL download."""
    if os.path.exists(local_path):
        print(f"  ✅  {description}: local copy present ({os.path.getsize(local_path)/1e6:.1f} MB)")
        return

    # 1. Try Drive
    if load_from_drive(local_path, drive_rel):
        return

    # 2. Download
    print(f"  ⬇️   Downloading {description}...")
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    result = subprocess.run(['curl', '-L', '-o', local_path, fallback_url],
                            capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✅  {description}: downloaded ({os.path.getsize(local_path)/1e6:.1f} MB)")
        save_to_drive(local_path, drive_rel)   # cache to Drive for next time
    else:
        print(f"  ❌  {description}: FAILED\n{result.stderr[:300]}")

print("\n📥  Checking raw data...")

ensure_raw_file(
    local_path  = "data/raw/nyc_charging.csv",
    drive_rel   = "data/raw/nyc_charging.csv",
    fallback_url= "https://data.cityofnewyork.us/api/views/kj7g-u4gp/rows.csv?accessType=DOWNLOAD",
    description = "NYC PlugNYC EV Charging Sessions (2021–2026)"
)

ensure_raw_file(
    local_path  = "data/raw/Automated_Traffic_Volume_Counts_20260415.csv",
    drive_rel   = "data/raw/Automated_Traffic_Volume_Counts_20260415.csv",
    fallback_url= "https://data.cityofnewyork.us/api/views/7ym2-wayt/rows.csv?accessType=DOWNLOAD",
    description = "NYC ATVC Hourly Traffic (2021–2026)"
)

print("\n📦  data/raw/ contents:")
for f in os.listdir("data/raw"):
    fp = os.path.join("data/raw", f)
    print(f"  {fp}  ({os.path.getsize(fp)/1e6:.2f} MB)")


## §3 · NYC Weather — Open-Meteo Historical API
Fetches **hourly** weather for **NYC (lat=40.7128, lon=-74.0060)** from **2021-07-31 → 2026-04-13** — perfectly bounding the EV charging dataset.

No API key required. Cached to Drive after first fetch.


In [ ]:
import pandas as pd
import numpy as np

WEATHER_LOCAL = "data/processed/weather_features.parquet"
WEATHER_DRIVE = "data/processed/weather_features.parquet"

NYC_LAT       = 40.7128
NYC_LON       = -74.0060
WEATHER_START = "2021-07-31"
WEATHER_END   = "2026-04-13"

os.makedirs("data/processed", exist_ok=True)

# ── Priority: local → Drive → fetch ──────────────────────────────────────────
if os.path.exists(WEATHER_LOCAL):
    df_weather = pd.read_parquet(WEATHER_LOCAL)
    print(f"✅  Weather: local cache found ({len(df_weather):,} rows)")
elif load_from_drive(WEATHER_LOCAL, WEATHER_DRIVE):
    df_weather = pd.read_parquet(WEATHER_LOCAL)
    print(f"✅  Weather: loaded from Drive ({len(df_weather):,} rows)")
else:
    print("📡  Fetching NYC hourly weather from Open-Meteo...")
    print(f"    lat={NYC_LAT}, lon={NYC_LON} | {WEATHER_START} → {WEATHER_END}")
    try:
        import openmeteo_requests, requests_cache
        from retry_requests import retry

        cache_session = requests_cache.CachedSession('.openmeteo_cache', expire_after=-1)
        retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
        client = openmeteo_requests.Client(session=retry_session)

        params = {
            "latitude"  : NYC_LAT,
            "longitude" : NYC_LON,
            "start_date": WEATHER_START,
            "end_date"  : WEATHER_END,
            "hourly"    : ["temperature_2m", "precipitation", "cloudcover"],
            "timezone"  : "America/New_York"
        }
        resp    = client.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
        hourly  = resp.Hourly()
        n       = hourly.Variables(0).ValuesAsNumpy().shape[0]

        ts = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True).tz_convert("America/New_York"),
            periods=n, freq="h"
        ).tz_localize(None)

        df_weather = pd.DataFrame({
            'timestamp'         : ts,
            'temperature_c'     : hourly.Variables(0).ValuesAsNumpy(),
            'precipitation_mm'  : hourly.Variables(1).ValuesAsNumpy(),
            'solar_availability': np.clip(1.0 - hourly.Variables(2).ValuesAsNumpy() / 100.0, 0.0, 1.0),
        }).dropna()

        df_weather['date'] = df_weather['timestamp'].dt.strftime('%Y-%m-%d')
        df_weather['hour'] = df_weather['timestamp'].dt.hour
        df_weather = df_weather[['date', 'hour', 'temperature_c', 'precipitation_mm', 'solar_availability']]

        df_weather.to_parquet(WEATHER_LOCAL, index=False)
        save_to_drive(WEATHER_LOCAL, WEATHER_DRIVE)
        print(f"✅  Weather fetched, cached locally & on Drive: {df_weather.shape}")

    except Exception as e:
        print(f"⚠️   Open-Meteo failed ({e}) — building synthetic NYC placeholder...")
        dates = pd.date_range(WEATHER_START, WEATHER_END, freq='D')
        rows  = []
        for d in dates:
            doy = d.timetuple().tm_yday
            for h in range(24):
                temp  = 14.5 + 12.5 * np.sin((doy - 80) * 2 * np.pi / 365) + 3 * np.sin(h * np.pi / 12)
                solar = max(0.0, np.sin((h - 6) * np.pi / 12))
                rows.append({'date': d.strftime('%Y-%m-%d'), 'hour': h,
                             'temperature_c': float(temp), 'precipitation_mm': 0.0,
                             'solar_availability': float(solar)})
        df_weather = pd.DataFrame(rows)
        df_weather.to_parquet(WEATHER_LOCAL, index=False)
        save_to_drive(WEATHER_LOCAL, WEATHER_DRIVE)
        print(f"✅  Synthetic NYC weather saved: {df_weather.shape}")

print(f"\n  Weather date range : {df_weather['date'].min()} → {df_weather['date'].max()}")
print(f"  Records            : {len(df_weather):,}")
print(df_weather.describe().round(3))


## §4 · NYC ATVC Traffic Index Preprocessing
**Schema**: `Yr, M, D, HH, MM, Vol` — vehicle counts at 15-min intervals across all NYC boroughs (2021–2026, 436k rows).

Produces two outputs — both cached to Drive:
- `traffic_tensor_24h.npy` — canonical 24h mean profile used as fallback
- `traffic_daily_24h.parquet` — per-day hourly profiles for day-accurate training


In [ ]:
import numpy as np
import pandas as pd

ATVC_PATH       = "data/raw/Automated_Traffic_Volume_Counts_20260415.csv"
TRAFFIC_NPY     = "data/processed/traffic_tensor_24h.npy"
TRAFFIC_DAILY   = "data/processed/traffic_daily_24h.parquet"

os.makedirs("data/processed", exist_ok=True)

# ── Priority: local → Drive → compute ────────────────────────────────────────
if os.path.exists(TRAFFIC_NPY) and os.path.exists(TRAFFIC_DAILY):
    traffic_24h = np.load(TRAFFIC_NPY)
    print(f"✅  Traffic: local cache found — {traffic_24h.shape}")
elif load_from_drive(TRAFFIC_NPY, TRAFFIC_NPY) and load_from_drive(TRAFFIC_DAILY, TRAFFIC_DAILY):
    traffic_24h = np.load(TRAFFIC_NPY)
    print(f"✅  Traffic: loaded from Drive — {traffic_24h.shape}")
else:
    print("🚦  Building NYC hourly traffic index from ATVC...")
    try:
        atvc = pd.read_csv(ATVC_PATH,
                           usecols=['Yr', 'M', 'D', 'HH', 'Vol'],
                           dtype={'Yr': int, 'M': int, 'D': int, 'HH': int, 'Vol': int})
        print(f"  Loaded {len(atvc):,} rows | Yr: {atvc['Yr'].min()}–{atvc['Yr'].max()}")

        atvc['date'] = (atvc['Yr'].astype(str) + '-' +
                        atvc['M'].astype(str).str.zfill(2) + '-' +
                        atvc['D'].astype(str).str.zfill(2))

        # Sum all segment volumes per (date, hour)
        hourly = atvc.groupby(['date', 'HH'])['Vol'].sum().reset_index()
        hourly.columns = ['date', 'hour', 'total_vol']

        pivot  = hourly.pivot_table(index='date', columns='hour',
                                    values='total_vol', fill_value=0)
        pivot  = pivot.reindex(columns=range(24), fill_value=0)
        daily  = pivot.values.astype(np.float32)

        # Global normalise [0, 1]
        lo, hi     = daily.min(), daily.max()
        daily_norm = (daily - lo) / (hi - lo + 1e-8)

        # Per-day parquet
        daily_df = pd.DataFrame(daily_norm,
                                index=pivot.index,
                                columns=[f"h{h:02d}" for h in range(24)])
        daily_df.index.name = 'date'
        daily_df.reset_index().to_parquet(TRAFFIC_DAILY, index=False)

        # Canonical 24h mean
        traffic_24h = daily_norm.mean(axis=0).astype(np.float32)
        np.save(TRAFFIC_NPY, traffic_24h)

        # Push both to Drive
        save_to_drive(TRAFFIC_NPY,   TRAFFIC_NPY)
        save_to_drive(TRAFFIC_DAILY, TRAFFIC_DAILY)
        print(f"✅  ATVC profile built & saved to Drive — {traffic_24h.shape}")

    except Exception as e:
        print(f"⚠️   ATVC parse failed ({e}) — NYC-tuned FHWA fallback...")
        fhwa = np.array([0.018,0.013,0.010,0.009,0.011,0.020,
                         0.040,0.078,0.095,0.075,0.060,0.055,
                         0.058,0.055,0.058,0.070,0.088,0.100,
                         0.090,0.068,0.048,0.036,0.027,0.022], dtype=np.float32)
        traffic_24h = (fhwa - fhwa.min()) / (fhwa.max() - fhwa.min())
        np.save(TRAFFIC_NPY, traffic_24h)
        save_to_drive(TRAFFIC_NPY, TRAFFIC_NPY)
        print(f"✅  NYC FHWA fallback profile saved — {traffic_24h.shape}")

print("\n  Hourly NYC Traffic Index:")
print("  " + "─"*50)
for h in range(24):
    bar = "█" * int(traffic_24h[h] * 35)
    print(f"  {h:02d}:00  {traffic_24h[h]:.3f}  {bar}")


## §5 · High-Performance Bootstrap
Groups 233,865 NYC charging sessions by historical day → bootstraps **5,000 synthetic days** — sessions distributed across 32 IEEE-33 bus nodes weighted by NYC ATVC `traffic_index[hour]`.

Output saved both locally and to Drive. Subsequent runtimes skip this step entirely.


In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

TRAIN_LOCAL  = "data/processed/train_data.parquet"
TRAIN_DRIVE  = "data/processed/train_data.parquet"
NUM_NODES    = 32
NUM_SCENARIOS= 5000

os.makedirs("data/processed", exist_ok=True)

# ── Priority: local → Drive → compute ────────────────────────────────────────
if os.path.exists(TRAIN_LOCAL):
    df_check = pd.read_parquet(TRAIN_LOCAL)
    print(f"✅  Bootstrap: local cache — {df_check['date'].nunique()} days × "
          f"{df_check['node_id'].nunique()} nodes = {len(df_check):,} rows")
    del df_check
elif load_from_drive(TRAIN_LOCAL, TRAIN_DRIVE):
    df_check = pd.read_parquet(TRAIN_LOCAL)
    print(f"✅  Bootstrap: loaded from Drive — {len(df_check):,} rows")
    del df_check
else:
    print("⚙️   [1/5] Loading NYC PlugNYC sessions...")
    df = pd.read_csv("data/raw/nyc_charging.csv")
    print(f"  Raw rows : {len(df):,}")

    # NYC schema: Date (MM/DD/YYYY), Connected Time (HH:MM:SS), Disconnected Time
    df['start_dt'] = pd.to_datetime(
        df['Date'].astype(str) + ' ' + df['Connected Time'].astype(str).str[:8],
        format='%m/%d/%Y %H:%M:%S', errors='coerce'
    )
    df['end_dt'] = pd.to_datetime(
        df['Date'].astype(str) + ' ' + df['Disconnected Time'].astype(str).str[:8],
        format='%m/%d/%Y %H:%M:%S', errors='coerce'
    )
    df['kWh']     = pd.to_numeric(df['Energy Provided (kWh)'],   errors='coerce')
    df['dur_min'] = pd.to_numeric(df['Charge Duration (min)'],   errors='coerce')

    df = df.dropna(subset=['start_dt', 'kWh', 'dur_min'])
    df = df[(df['kWh'] > 0) & (df['dur_min'] > 0)]

    # Cross-midnight correction
    mask = df['end_dt'] < df['start_dt']
    df.loc[mask, 'end_dt'] += pd.Timedelta(days=1)

    df['date']       = df['start_dt'].dt.date
    df['hour']       = df['start_dt'].dt.hour
    df['duration_h'] = (df['dur_min'] / 60.0).clip(lower=0.083)
    df['avg_kw']     = (df['kWh'] / df['duration_h']).clip(upper=350.0)

    print(f"  ✅  [2/5] Clean sessions: {len(df):,} | {df['date'].min()} → {df['date'].max()}")

    daily_groups    = dict(tuple(df.groupby('date')))
    historical_days = list(daily_groups.keys())
    print(f"  ✅  [3/5] Historical days: {len(historical_days)}")

    traffic_24h = np.load(TRAFFIC_NPY)
    hour_node_probs = np.zeros((24, NUM_NODES), dtype=np.float32)
    for h in range(24):
        hour_node_probs[h] = np.full(NUM_NODES, traffic_24h[h] / NUM_NODES)
    hour_node_probs /= hour_node_probs.sum(axis=1, keepdims=True)
    print(f"  ✅  [4/5] Node prob matrix: {hour_node_probs.shape}")
    print(f"  ⚙️   [5/5] Bootstrapping {NUM_SCENARIOS:,} scenarios...")

    rng          = np.random.default_rng(42)
    sampled_days = [historical_days[i] for i in rng.integers(0, len(historical_days), NUM_SCENARIOS)]
    nodes_arr    = np.arange(NUM_NODES)
    base_date    = pd.Timestamp("2021-07-31")

    TOTAL = NUM_SCENARIOS * 24 * NUM_NODES
    out_dates = np.empty(TOTAL, dtype='U10')
    out_hours = np.empty(TOTAL, dtype=np.int8)
    out_nodes = np.empty(TOTAL, dtype=np.int8)
    out_kw    = np.zeros(TOTAL,  dtype=np.float32)

    ptr = 0
    for i in tqdm(range(NUM_SCENARIOS), desc="Bootstrap scenarios"):
        sessions     = daily_groups[sampled_days[i]]
        gen_date_str = (base_date + pd.Timedelta(days=i)).strftime('%Y-%m-%d')
        day_demand   = np.zeros((24, NUM_NODES), dtype=np.float32)
        for _, row in sessions.iterrows():
            sh   = int(row['hour'])
            dur  = max(1, int(round(row['duration_h'])))
            kw   = float(row['avg_kw'])
            node = rng.choice(nodes_arr, p=hour_node_probs[sh])
            for offset in range(dur):
                day_demand[(sh + offset) % 24, node] += kw
        bs = 24 * NUM_NODES
        out_dates[ptr:ptr+bs] = gen_date_str
        out_hours[ptr:ptr+bs] = np.repeat(np.arange(24), NUM_NODES)
        out_nodes[ptr:ptr+bs] = np.tile(nodes_arr, 24)
        out_kw[ptr:ptr+bs]    = day_demand.flatten()
        ptr += bs

    final_df = pd.DataFrame({
        'date':      out_dates,
        'hour':      out_hours,
        'node_id':   [f"node_{n:02d}" for n in out_nodes],
        'demand_kw': out_kw,
    })
    final_df.to_parquet(TRAIN_LOCAL, index=False)
    save_to_drive(TRAIN_LOCAL, TRAIN_DRIVE)
    print(f"\n✅  Bootstrap complete → {TRAIN_LOCAL} & Drive")
    print(f"   Shape: {final_df.shape} | kW: [{final_df['demand_kw'].min():.2f}, {final_df['demand_kw'].max():.2f}]")


## §6 · Hyperparameter Configuration & Hardware Detection

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import r2_score, mean_absolute_error
import math, os, time, json, warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'figure.dpi': 130, 'font.family': 'DejaVu Sans', 'axes.grid': True})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️   Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"    GPU : {torch.cuda.get_device_name(0)}")
    print(f"    VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

class CFG:
    # Geography
    CITY        = "New York City, NY"
    LAT         = 40.7128
    LON         = -74.0060
    DATA_START  = "2021-07-31"
    DATA_END    = "2026-04-13"

    # Grid
    NUM_NODES        = 32
    SEQ_LEN          = 24
    NUM_WEATHER_FEAT = 4        # temp, precip, solar, traffic
    NUM_FEATURES     = NUM_NODES + NUM_WEATHER_FEAT   # 36
    COND_DIM         = 6        # [temp_anomaly, ev_mult, solar, weekend, holiday, traffic]

    # Model
    TCN_CHANNELS   = [128, 256, 256, 256]
    KERNEL_SIZE    = 3
    DROPOUT        = 0.15
    LATENT_DIM     = 128
    DECODER_HIDDEN = 512

    # Training
    EPOCHS           = 200
    BATCH_SIZE       = 64
    LEARNING_RATE    = 2e-4
    GRAD_CLIP_NORM   = 1.0
    LOG_EVERY        = 5
    SAVE_EVERY       = 10
    KLD_WARMUP_EPOCHS= 50
    KLD_MAX          = 0.10
    PHYSICS_EPOCH_ON = 51
    LAMBDA_VOLT      = 1000.0
    LAMBDA_THERMAL   = 500.0
    LAMBDA_XFMR      = 800.0

    # Paths (local defaults — overridden to Drive below)
    CHECKPOINT_DIR = "output/checkpoints"
    BEST_CKPT      = "output/best_model.pt"
    FINAL_CKPT     = "output/gcvae_final.pt"
    LOG_FILE       = "output/training_log.jsonl"
    METRICS_FILE   = "output/metrics_history.json"

# ── Redirect all outputs to Drive when on Colab ───────────────────────────────
if DRIVE_ROOT:
    CFG.CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
    CFG.BEST_CKPT      = os.path.join(DRIVE_ROOT, "best_model.pt")
    CFG.FINAL_CKPT     = os.path.join(DRIVE_ROOT, "gcvae_final.pt")
    CFG.LOG_FILE       = os.path.join(DRIVE_ROOT, "training_log.jsonl")
    CFG.METRICS_FILE   = os.path.join(DRIVE_ROOT, "metrics_history.json")
    print(f"\n📂  Colab — all outputs → {DRIVE_ROOT}")
else:
    print("\n💻  Local mode — outputs → output/")

os.makedirs(CFG.CHECKPOINT_DIR, exist_ok=True)
os.makedirs("output", exist_ok=True)

print(f"\n{'='*55}")
print(f"  EVolvAI GCD-VAE | {CFG.CITY}")
print(f"  Data     : {CFG.DATA_START} → {CFG.DATA_END}")
print(f"  Features : {CFG.NUM_FEATURES} ({CFG.NUM_NODES} nodes + {CFG.NUM_WEATHER_FEAT} weather)")
print(f"  Cond dim : {CFG.COND_DIM}")
print(f"  Epochs   : {CFG.EPOCHS} | Physics @ {CFG.PHYSICS_EPOCH_ON}")
print(f"{'='*55}")


## §7 · PyTorch Dataset — Fuse All Three Data Sources
Each training sample = one synthetic day fusing:
1. **32 EV demand channels** — from bootstrap (NYC sessions)
2. **4 weather channels** — `temp_c, precip_mm, solar_availability` (Open-Meteo NYC) + `traffic_index` (NYC ATVC)
3. **6-D condition vector** — derived from calendar date + daily mean traffic


In [ ]:
import datetime

def _date_to_condition(date_str: str, traffic_val: float = 0.5) -> list:
    """6-D condition vector from calendar date. Fully deterministic."""
    try:
        d   = datetime.date.fromisoformat(date_str)
        doy = d.timetuple().tm_yday
        solar   = float(np.clip(0.5 + 0.5 * np.sin((doy - 80) * 2 * np.pi / 365), 0.0, 1.0))
        weekend = float(d.weekday() >= 5)
    except Exception:
        return [0.0, 1.0, 0.5, 0.0, 0.0, 0.5]
    return [0.0, 1.0, solar, weekend, 0.0, float(traffic_val)]


class EVDemandDataset(Dataset):
    """
    Fuses all three NYC data sources into (x:[36,24], cond:[6]) per day:
      ch 0–31  : EV demand_kw  (bootstrapped NYC sessions)
      ch 32    : temperature_c (Open-Meteo NYC)
      ch 33    : precipitation_mm (Open-Meteo NYC)
      ch 34    : solar_availability (Open-Meteo NYC)
      ch 35    : traffic_index (NYC ATVC)
    """

    def __init__(self):
        print("\n⚙️   Loading EVDemandDataset...")

        # 1. EV demand
        df_ev = pd.read_parquet("data/processed/train_data.parquet")
        print(f"  EV demand  : {len(df_ev):,} rows | {df_ev['date'].nunique():,} days")
        pivot = df_ev.pivot_table(index=['date','hour'], columns='node_id',
                                   values='demand_kw', aggfunc='sum', fill_value=0.0)
        pivot = pivot.sort_index().reindex(
            columns=[f"node_{i:02d}" for i in range(CFG.NUM_NODES)], fill_value=0.0
        )

        # 2. Traffic — per-day profiles preferred
        traffic_24h = np.load("data/processed/traffic_tensor_24h.npy")
        daily_traffic = None
        if os.path.exists("data/processed/traffic_daily_24h.parquet"):
            dt = pd.read_parquet("data/processed/traffic_daily_24h.parquet").set_index('date')
            daily_traffic = dt
            print(f"  ATVC daily : {len(dt):,} date profiles available")

        # 3. Weather (Open-Meteo NYC)
        df_w = None
        for wf in ["data/processed/weather_features.parquet", "weather_data.parquet"]:
            if os.path.exists(wf):
                df_w = pd.read_parquet(wf)
                print(f"  Weather    : {wf}  ({len(df_w):,} rows)")
                break

        all_dates = sorted(pivot.index.get_level_values('date').unique().tolist())
        demand_list, weather_list, cond_list, valid_dates = [], [], [], []

        for date in all_dates:
            try:
                day_ev = pivot.loc[date]
                if len(day_ev) != CFG.SEQ_LEN:
                    continue
                demand_arr = day_ev.values.astype(np.float32)  # [24, 32]

                date_str = str(date)

                # Traffic channel (per-day if available)
                if daily_traffic is not None and date_str in daily_traffic.index:
                    day_traffic = daily_traffic.loc[date_str].values.astype(np.float32)
                else:
                    day_traffic = traffic_24h

                # Weather channels [24, 4]
                w_arr = np.zeros((CFG.SEQ_LEN, CFG.NUM_WEATHER_FEAT), dtype=np.float32)
                if df_w is not None:
                    day_w = df_w[df_w['date'] == date_str]
                    if len(day_w) >= CFG.SEQ_LEN:
                        for ci, col in enumerate(['temperature_c','precipitation_mm','solar_availability']):
                            if col in day_w.columns:
                                w_arr[:, ci] = day_w[col].values[:CFG.SEQ_LEN]
                w_arr[:, 3] = day_traffic   # ch 3 = ATVC traffic

                demand_list.append(demand_arr)
                weather_list.append(w_arr)
                cond_list.append(_date_to_condition(date_str, float(day_traffic.mean())))
                valid_dates.append(date)
            except Exception:
                continue

        demand_np  = np.stack(demand_list,  axis=0)   # [N, 24, 32]
        weather_np = np.stack(weather_list, axis=0)   # [N, 24, 4]

        def znorm(a):
            mu = a.mean(axis=(0,1), keepdims=True)
            sd = a.std(axis=(0,1),  keepdims=True)
            return (a - mu) / (sd + 1e-6)

        fused = np.concatenate([znorm(demand_np), znorm(weather_np)],
                                axis=-1).astype(np.float32)  # [N, 24, 36]

        self._data  = fused
        self._conds = np.array(cond_list, dtype=np.float32)
        self._dates = valid_dates

        print(f"  ✅  Dataset: {len(self):,} samples | shape={self._data.shape}")
        print(f"      Cond dim={self._conds.shape[1]} | {valid_dates[0]} → {valid_dates[-1]}")

    def __len__(self): return len(self._data)
    def __getitem__(self, idx):
        x    = torch.from_numpy(self._data[idx]).permute(1, 0)  # [36, 24]
        cond = torch.from_numpy(self._conds[idx])
        return x, cond

print("✅  EVDemandDataset class defined")


## §8 · GCD-VAE Model Architecture

In [ ]:
class CausalConv1d(nn.Conv1d):
    def __init__(self, in_ch, out_ch, ks, dilation=1, **kw):
        self._pad = (ks - 1) * dilation
        super().__init__(in_ch, out_ch, ks, padding=self._pad, dilation=dilation, **kw)
    def forward(self, x):
        out = super().forward(x)
        return out[:, :, :-self._pad] if self._pad else out

class TCNBlock(nn.Module):
    def __init__(self, n_in, n_out, ks, dilation, dropout):
        super().__init__()
        self.net  = nn.Sequential(
            CausalConv1d(n_in, n_out, ks, dilation), nn.LayerNorm([n_out, CFG.SEQ_LEN]), nn.GELU(), nn.Dropout(dropout),
            CausalConv1d(n_out,n_out, ks, dilation), nn.LayerNorm([n_out, CFG.SEQ_LEN]), nn.GELU(), nn.Dropout(dropout),
        )
        self.skip = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else nn.Identity()
    def forward(self, x): return F.gelu(self.net(x) + self.skip(x))

class TemporalConvNet(nn.Module):
    def __init__(self, n_in, channels, ks, dropout):
        super().__init__()
        layers, prev = [], n_in
        for i, ch in enumerate(channels):
            layers.append(TCNBlock(prev, ch, ks, 2**i, dropout))
            prev = ch
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

class GenerativeCounterfactualVAE(nn.Module):
    def __init__(self):
        super().__init__()
        flat = CFG.SEQ_LEN * CFG.TCN_CHANNELS[-1]
        self.enc_tcn   = TemporalConvNet(CFG.NUM_FEATURES, CFG.TCN_CHANNELS, CFG.KERNEL_SIZE, CFG.DROPOUT)
        self.attention = nn.MultiheadAttention(CFG.TCN_CHANNELS[-1], num_heads=8, dropout=CFG.DROPOUT, batch_first=True)
        self.attn_norm = nn.LayerNorm(CFG.TCN_CHANNELS[-1])
        self.fc_mu     = nn.Linear(flat, CFG.LATENT_DIM)
        self.fc_logvar = nn.Linear(flat, CFG.LATENT_DIM)
        self.dec_fc    = nn.Sequential(
            nn.Linear(CFG.LATENT_DIM + CFG.COND_DIM, CFG.DECODER_HIDDEN), nn.GELU(), nn.Dropout(CFG.DROPOUT),
            nn.Linear(CFG.DECODER_HIDDEN, flat), nn.GELU()
        )
        self.dec_tcn   = TemporalConvNet(CFG.TCN_CHANNELS[-1], [256, 128, CFG.NUM_FEATURES], CFG.KERNEL_SIZE, CFG.DROPOUT)

    def encode(self, x):
        h = self.enc_tcn(x)                           # [B,256,24]
        h_t = h.permute(0,2,1)                        # [B,24,256]
        a, _ = self.attention(h_t, h_t, h_t)
        h_flat = self.attn_norm(h_t + a).flatten(1)
        return self.fc_mu(h_flat), self.fc_logvar(h_flat)

    def reparameterize(self, mu, logvar):
        return mu + torch.randn_like(mu) * (0.5 * logvar).exp()

    def decode(self, z, cond):
        h = self.dec_fc(torch.cat([z, cond], -1))
        h = h.view(-1, CFG.TCN_CHANNELS[-1], CFG.SEQ_LEN)
        return self.dec_tcn(h)                        # [B,36,24]

    def forward(self, x, cond):
        mu, lv = self.encode(x)
        return self.decode(self.reparameterize(mu, lv), cond), mu, lv

m = GenerativeCounterfactualVAE()
total = sum(p.numel() for p in m.parameters())
print(f"✅  GCD-VAE | {total/1e6:.2f}M params | Latent={CFG.LATENT_DIM} | Heads=8")
del m


## §9 · LinDistFlow Physics Engine

In [ ]:
from data_pipeline.physics_penalty_engine import physics_penalty_engine
from generative_core.physics_loss import LinDistFlowLoss

lindistflow = LinDistFlowLoss(DEVICE)
lindistflow.eval()

print("✅  LinDistFlowLoss ready")
print(f"   V bounds    : [{lindistflow.v_min:.2f}, {lindistflow.v_max:.2f}] p.u.")
print(f"   I limit     : {lindistflow.i_lim_pu:.2f} p.u.")
print(f"   Transformer : {lindistflow.xfmr_kva:.0f} kVA")

dummy = torch.zeros(1, 24, 32, device=DEVICE)
v_p, t_p, x_p = lindistflow(dummy)
assert v_p.item() < 1e-5, "FAIL: zero load must give zero penalty"
print("   ✅  Sanity PASSED (zero load → zero penalty)")


## §10 · Training Engine

In [ ]:
from generative_core.physics_loss import LinDistFlowLoss

def compute_r2_mae(t, p):
    tf, pf = t.reshape(-1), p.reshape(-1)
    return float(r2_score(tf, pf)), float(mean_absolute_error(tf, pf))


def train_evolvai():
    model     = GenerativeCounterfactualVAE().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-5)
    physics   = LinDistFlowLoss(DEVICE)
    dataset   = EVDemandDataset()
    loader    = DataLoader(dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=(DEVICE.type=='cuda'), prefetch_factor=2)

    print(f"\n{'='*65}")
    print(f"  EVolvAI GCD-VAE Training — {CFG.CITY}")
    print(f"  Data   : {CFG.DATA_START} → {CFG.DATA_END}")
    print(f"  Samples: {len(dataset):,} | Batches/epoch: {len(loader)}")
    print(f"  Output : {CFG.CHECKPOINT_DIR}")
    print(f"{'='*65}\n")

    history = {k: [] for k in ['epoch','total_loss','recon_loss','kld_loss','physics_loss',
                                'r2','mae','kld_weight','lr','v_penalty','t_penalty','x_penalty','epoch_time_s']}
    best_r2, start_epoch = -np.inf, 1

    latest_ckpt = os.path.join(CFG.CHECKPOINT_DIR, "latest.pt")
    if os.path.exists(latest_ckpt):
        print("🔄  Resuming from checkpoint...")
        ck = torch.load(latest_ckpt, map_location=DEVICE)
        model.load_state_dict(ck['model_state_dict'])
        optimizer.load_state_dict(ck['optimizer_state_dict'])
        scheduler.load_state_dict(ck['scheduler_state_dict'])
        start_epoch = ck['epoch'] + 1
        history, best_r2 = ck.get('history', history), ck.get('best_r2', best_r2)
        print(f"   Resumed @ epoch {ck['epoch']} (best R²={best_r2:.4f})")

    log_fh = open(CFG.LOG_FILE, 'a')

    for epoch in range(start_epoch, CFG.EPOCHS + 1):
        model.train()
        t0 = time.time()
        kld_w   = min(CFG.KLD_MAX, (epoch / CFG.KLD_WARMUP_EPOCHS) * CFG.KLD_MAX)
        phys_on = epoch >= CFG.PHYSICS_EPOCH_ON

        e_total=e_recon=e_kld=e_phys=e_v=e_t=e_x=0.0
        all_t, all_p, n_b = [], [], 0

        for bx, bc in loader:
            x, cond = bx.to(DEVICE), bc.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar = model(x, cond)

            recon_loss = F.mse_loss(recon, x, reduction='mean')
            kld_loss   = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss       = recon_loss + kld_w * kld_loss

            v_pen = t_pen = x_pen = torch.tensor(0.0, device=DEVICE)
            if phys_on:
                ev_pow = recon[:, :CFG.NUM_NODES, :].permute(0,2,1).contiguous()
                v_pen, t_pen, x_pen = physics(ev_pow)
                pl   = CFG.LAMBDA_VOLT*v_pen + CFG.LAMBDA_THERMAL*t_pen + CFG.LAMBDA_XFMR*x_pen
                loss = loss + pl
                e_phys+=pl.item(); e_v+=v_pen.item(); e_t+=t_pen.item(); e_x+=x_pen.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP_NORM)
            optimizer.step()
            e_total+=loss.item(); e_recon+=recon_loss.item(); e_kld+=kld_loss.item()
            all_t.append(x.detach().cpu().numpy())
            all_p.append(recon.detach().cpu().numpy())
            n_b += 1

        scheduler.step()
        nb = max(n_b, 1)
        r2, mae = compute_r2_mae(np.concatenate(all_t), np.concatenate(all_p))
        et = time.time()-t0
        lr = optimizer.param_groups[0]['lr']

        for k, v in zip(history.keys(),
            [epoch, e_total/nb, e_recon/nb, e_kld/nb, e_phys/nb,
             r2, mae, kld_w, lr, e_v/nb, e_t/nb, e_x/nb, et]):
            history[k].append(v)

        log_fh.write(json.dumps({'epoch':epoch,'total_loss':round(e_total/nb,6),
            'recon_loss':round(e_recon/nb,6),'kld_loss':round(e_kld/nb,6),
            'physics_loss':round(e_phys/nb,6),'r2':round(r2,6),'mae':round(mae,6),
            'kld_weight':round(kld_w,6),'lr':round(lr,8),'physics_on':phys_on,
            'v_penalty':round(e_v/nb,8),'t_penalty':round(e_t/nb,8),'x_penalty':round(e_x/nb,8),
            'epoch_time_s':round(et,2)}) + '\n'); log_fh.flush()

        if epoch % CFG.LOG_EVERY == 0 or epoch == 1:
            pt = f"| Phys: {e_phys/nb:.4f}" if phys_on else "| Phys: [WARMUP]"
            print(f"Ep {epoch:>4}/{CFG.EPOCHS} | Loss:{e_total/nb:.5f} | Recon:{e_recon/nb:.5f} "
                  f"| KLD:{e_kld/nb:.5f} {pt} | R²:{r2:+.4f} | MAE:{mae:.4f} | lr:{lr:.2e} | {et:.1f}s")

        def _ckpt(path):
            torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(),
                'scheduler_state_dict':scheduler.state_dict(),
                'history':history,'best_r2':best_r2,
                'cfg':{k:v for k,v in vars(CFG).items() if not k.startswith('_')}}, path)

        if epoch % CFG.SAVE_EVERY == 0 or epoch == CFG.EPOCHS:
            _ckpt(latest_ckpt)
            _ckpt(os.path.join(CFG.CHECKPOINT_DIR, f"epoch_{epoch:04d}.pt"))
            print(f"   💾  Checkpoint → Drive @ epoch {epoch}")

        if r2 > best_r2:
            best_r2 = r2; _ckpt(CFG.BEST_CKPT)
            print(f"   ⭐  New best R²={best_r2:.4f}")

    log_fh.close()
    _ckpt(CFG.FINAL_CKPT)
    with open(CFG.METRICS_FILE, 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\n{'='*65}")
    print(f"  Training complete! Best R²={best_r2:.4f}")
    print(f"{'='*65}")
    return model, history, dataset


trained_model, history, dataset = train_evolvai()


## §11 · Training Diagnostics & Publication Figures

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle(f'EVolvAI GCD-VAE — Training Diagnostics ({CFG.CITY})', fontsize=15, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.36)
eps = history['epoch']

ax = fig.add_subplot(gs[0,0])
ax.plot(eps,history['total_loss'],'#e74c3c',lw=2,label='Total')
ax.plot(eps,history['recon_loss'],'#3498db',lw=1.5,ls='--',label='Recon')
ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.legend(); ax.set_yscale('log')

ax = fig.add_subplot(gs[0,1])
ax.plot(eps,history['kld_loss'],'#9b59b6',lw=2,label='KLD')
ax.plot(eps,history['kld_weight'],'#f39c12',lw=1.5,ls=':',label='Weight')
ax.axvline(CFG.PHYSICS_EPOCH_ON,color='red',ls='--',alpha=0.5,label=f'Physics@{CFG.PHYSICS_EPOCH_ON}')
ax.set_title('KLD & Annealing'); ax.legend(fontsize=7)

ax = fig.add_subplot(gs[0,2])
if any(v>0 for v in history['physics_loss']):
    ax.plot(eps,history['physics_loss'],'#e74c3c',lw=2,label='Total')
    ax.plot(eps,history['v_penalty'],'#3498db',lw=1.5,ls='--',label='Voltage')
    ax.plot(eps,history['t_penalty'],'#2ecc71',lw=1.5,ls='--',label='Thermal')
    ax.plot(eps,history['x_penalty'],'#f39c12',lw=1.5,ls='--',label='Xfmr')
else:
    ax.text(0.5,0.5,f'Warm-up\n(Physics @{CFG.PHYSICS_EPOCH_ON})',ha='center',va='center',transform=ax.transAxes)
ax.set_title('LinDistFlow Penalties'); ax.legend(fontsize=7)

ax = fig.add_subplot(gs[1,0])
ax.plot(eps,history['r2'],'#27ae60',lw=2)
ax.axhline(0.9,color='green',ls=':',alpha=0.5,label='R²=0.9 target')
ax.fill_between(eps,history['r2'],alpha=0.15,color='#27ae60')
ax.set_title('R²'); ax.legend()

ax = fig.add_subplot(gs[1,1])
ax.plot(eps,history['mae'],'#e67e22',lw=2)
ax.fill_between(eps,history['mae'],alpha=0.15,color='#e67e22')
ax.set_title('MAE (z-score)')

ax = fig.add_subplot(gs[1,2])
ax.semilogy(eps,history['lr'],'#8e44ad',lw=2)
ax.set_title('LR Cosine Decay')

ax = fig.add_subplot(gs[2,:2])
trained_model.eval()
ls = DataLoader(dataset,batch_size=8,shuffle=False)
xs,cs = next(iter(ls))
with torch.no_grad(): rs,_,_ = trained_model(xs.to(DEVICE),cs.to(DEVICE))
for k in range(4):
    ax.plot(xs[k,0,:].numpy(),alpha=0.6,lw=1.5,label='True' if k==0 else '')
    ax.plot(rs[k,0,:].cpu().numpy(),alpha=0.6,lw=1.5,ls='--',label='Recon' if k==0 else '')
ax.set_title('Sample Reconstructions — Node 0'); ax.legend()

ax = fig.add_subplot(gs[2,2])
ax.bar(eps,history['epoch_time_s'],color='#7f8c8d',alpha=0.7)
ax.set_title('Time/Epoch (s)')

FIG_PATH = 'output/training_diagnostics.png'
plt.savefig(FIG_PATH,bbox_inches='tight',dpi=150)
plt.show()
save_to_drive(FIG_PATH, FIG_PATH)
print("✅  Figure saved locally & pushed to Drive")


## §12 · Final Metrics Report

In [ ]:
best_model = GenerativeCounterfactualVAE().to(DEVICE)
ck = torch.load(CFG.BEST_CKPT, map_location=DEVICE)
best_model.load_state_dict(ck['model_state_dict'])
best_model.eval()

eval_loader = DataLoader(dataset, batch_size=128, shuffle=False)
all_true, all_pred = [], []
with torch.no_grad():
    for bx, bc in eval_loader:
        r, _, _ = best_model(bx.to(DEVICE), bc.to(DEVICE))
        all_true.append(bx.numpy()); all_pred.append(r.cpu().numpy())

true_np = np.concatenate(all_true)
pred_np = np.concatenate(all_pred)
r2f  = r2_score(true_np.flatten(), pred_np.flatten())
maef = mean_absolute_error(true_np.flatten(), pred_np.flatten())
zp   = np.mean(np.abs(pred_np[:, :CFG.NUM_NODES, :]) < 1e-4) * 100
tp   = sum(p.numel() for p in best_model.parameters())

print("="*60)
print("  EVolvAI GCD-VAE — Final Evaluation")
print("="*60)
print(f"  City           : {CFG.CITY}")
print(f"  Data range     : {CFG.DATA_START} → {CFG.DATA_END}")
print(f"  Parameters     : {tp:,} (~{tp/1e6:.1f}M)")
print(f"  Samples        : {len(dataset):,}")
print(f"  Final R²       : {r2f:+.4f}")
print(f"  Final MAE      : {maef:.4f} (z-score)")
print(f"  Best R²        : {max(history['r2']):.4f}")
print(f"  Zero-output %  : {zp:.2f}%  (target <2%)")
print(f"  Physics λ      : V={CFG.LAMBDA_VOLT} | T={CFG.LAMBDA_THERMAL} | X={CFG.LAMBDA_XFMR}")
print("="*60)

summary = {
    'city': CFG.CITY, 'data_start': CFG.DATA_START, 'data_end': CFG.DATA_END,
    'total_params': tp, 'r2_final': round(r2f,6), 'mae_final': round(maef,6),
    'zero_pct': round(zp,4), 'best_r2': round(max(history['r2']),6),
    'best_epoch': history['epoch'][history['r2'].index(max(history['r2']))],
    'n_samples': len(dataset), 'epochs_run': len(history['epoch']),
}
EVAL_PATH = 'output/eval_summary.json'
with open(EVAL_PATH, 'w') as f: json.dump(summary, f, indent=2)
save_to_drive(EVAL_PATH, EVAL_PATH)
print("\n✅  eval_summary.json saved locally & to Drive")


## §13 · NYC Counterfactual Demand Generation

In [ ]:
SCENARIOS = {
    "baseline"             : [0.0, 1.0, 1.0, 0.0, 0.0, 0.50],
    "nyc_winter_storm"     : [1.0, 2.5, 0.0, 0.0, 0.0, 0.85],
    "full_electrification" : [0.0, 3.0, 1.0, 0.0, 0.0, 0.65],
    "nyc_rush_hour"        : [0.0, 2.0, 0.5, 0.0, 0.0, 1.00],
    "nyc_summer_ev_peak"   : [0.5, 1.5, 1.0, 0.0, 0.0, 0.90],
}

os.makedirs("output", exist_ok=True)
best_model.eval()
with torch.no_grad():
    sx, _ = next(iter(DataLoader(dataset, batch_size=64, shuffle=True)))
    mu_z, _ = best_model.encode(sx.to(DEVICE))
    Z = mu_z.mean(dim=0, keepdim=True)   # [1, 128]

fig, axes = plt.subplots(1, len(SCENARIOS), figsize=(22, 4))
for ax, (name, cv) in zip(axes, SCENARIOS.items()):
    ct = torch.tensor([cv], dtype=torch.float32, device=DEVICE)
    with torch.no_grad(): out = best_model.decode(Z, ct)
    demand = out[0, :CFG.NUM_NODES, :].cpu().numpy()   # [32, 24]
    mean_d = demand.mean(0); peak_d = demand.max(0)
    np.save(f"output/scenario_{name}.npy", demand)
    save_to_drive(f"output/scenario_{name}.npy", f"output/scenario_{name}.npy")
    ax.fill_between(range(24), mean_d, alpha=0.35)
    ax.plot(range(24), peak_d, lw=1.5, ls='--', label='Peak')
    ax.plot(range(24), mean_d, lw=2, label='Mean')
    ax.set_title(name.replace('_',' ').title(), fontsize=9, fontweight='bold')
    ax.set_xlabel('Hour'); ax.legend(fontsize=6)
    print(f"  ✅  {name:30s}  peak={peak_d.max():.3f}  mean={mean_d.mean():.3f}")

CF_PATH = 'output/counterfactual_scenarios.png'
plt.suptitle(f'EVolvAI NYC Counterfactual Scenarios (Fixed Z)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(CF_PATH, bbox_inches='tight', dpi=150)
plt.show()
save_to_drive(CF_PATH, CF_PATH)
print("\n✅  All scenarios saved locally & to Drive")


## §14 · Final Drive Sync
Pushes any remaining local outputs to Drive. Safe to re-run at any time.


In [ ]:
print("🔄  Final Drive sync...\n")

# Processed data
for rel in [
    "data/processed/weather_features.parquet",
    "data/processed/traffic_tensor_24h.npy",
    "data/processed/traffic_daily_24h.parquet",
    "data/processed/train_data.parquet",
]:
    if os.path.exists(rel):
        save_to_drive(rel, rel)

# Output artifacts
for rel in [
    "output/training_diagnostics.png",
    "output/counterfactual_scenarios.png",
    "output/eval_summary.json",
    "output/training_log.jsonl",
    "output/metrics_history.json",
]:
    if os.path.exists(rel):
        save_to_drive(rel, rel)

print("\n✅  Drive sync complete.")
if DRIVE_ROOT:
    print(f"   All data lives at: {DRIVE_ROOT}")
    print("   Next runtime: just run §1 (mount Drive) → everything loads from cache.")
